### Data Structures

#### 1. Lists vs Tuples vs Sets vs Dicts

In [32]:
# List
print(f'{"LIST":>20}\n{"="*40}')
session_ids = ['S001', 'S002', 'S003']
session_ids.append('S004')
print(f'Append: {session_ids}')
session_ids[0] = 'S000'
print(f'Mutable: {session_ids}')

# tuple - ordered, immutable, allows duplciates
# use for fixed data that shouldnt change (coords, RGB, config)
print(f'\n{"TUPLE":>20}\n{"="*40}')
sensor_axes = ('acc_x', 'acc_y', 'acc_z') # wont be mutable
x, y, z = sensor_axes
print(f'Unpacked tuple x: {x}\nUnpacked tuple y: {y}\nUnpacked tuple z: {z}')

# SET - unordered, mutable, NO duplicates - o(1) lookup
# use for membership testing, deduplicates, overlap checks
print(f'\n{"SET":>20}\n{"="*40}')
train_pids = {'P001', 'P002', 'P003'}
test_pids = {'P004', 'P005'}
overlap = train_pids & test_pids
all_pids = train_pids | test_pids
print(f'Overlap: {overlap}\nUnion: {all_pids}')

# DICT - key:value, ordered, O(1) lookup\
print(f'\n{"DICT":>20}\n{"="*40}')
session_meta = {
    'S001': {'participant': 'P001', 'activity': 'walking', 'rows': 3000},
    'S002': {'participant': 'P002', 'activity': 'running', 'rows': 2800},
}
print(f"Session Meta: {session_meta['S001']['activity']}")
print(session_meta.get('S999', 'not found'))  # 'not found' — safe get

                LIST
Append: ['S001', 'S002', 'S003', 'S004']
Mutable: ['S000', 'S002', 'S003', 'S004']

               TUPLE
Unpacked tuple x: acc_x
Unpacked tuple y: acc_y
Unpacked tuple z: acc_z

                 SET
Overlap: set()
Union: {'P001', 'P005', 'P002', 'P003', 'P004'}

                DICT
Session Meta: walking
not found


#### Decorators

##### Q: What is a decorator? Write a simple timing decorator.
A: A decorator is a function that takes a function and returns a modified version of it. Uses @syntax sugar. Common use cases: timing, logging, authentication, caching

In [10]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f'{func.__name__} took {elapsed:.4f}s')
        return result
    return wrapper

@timer
def process_session(n):
    return sum(range(n))

print(f'{process_session(1_000_000)}')

# functools.lru_cache - built in memoisation decorator
from functools import lru_cache

@lru_cache(maxsize=120)
def fibonacci(n):
    if n < 2: return n
    return fibonacci(n-1) + fibonacci(n-2)
print(fibonacci(50))

process_session took 0.0128s
499999500000
12586269025


#### Classes and OOP Basic

##### Q: Write a class representing a sensor session with methods to compute statistics.
A: Demonstrate __init__, instance methods, properties, and __repr__. Keep it practical — not academic.

In [ ]:
class SensorSession:
    def __init__(self, session_id, participation_id, data):
        self.session_id         = session_id
        self.participation_id   = participation_id
        self.data               = data
    
    @property
    def n_rows(self):
        return len(self.data)

    def summary(self):
        if not self.data:
            return {'error': 'no data'}
        acc_vals = [row['acc_x'] for row in self.data]
        return {
            'session_id'    :   self.session_id,
            'n_rows'        :   self.n_rows,
            'acc_mean'      :   sum(acc_vals) / len(acc_vals),
            'acc_min'       :   min(acc_vals),
            'acc_max'       :   max(acc_vals),
        }
    
    def __repr__(self):
        return f'SensorSession({self.session_id}, {self.participation_id}, {self.n_rows} rows)'

# usage
data = [{'acc_x': 0.1}, {'acc_x': 0.5}, {'acc_x': -0.2}]
s = SensorSession('S001', 'P001', data)
print(s)
print(s.summary())

SensorSession(S001, P001, 3 rows)
{'session_id': 'S001', 'n_rows': 3, 'acc_mean': 0.13333333333333333, 'acc_min': -0.2, 'acc_max': 0.5}


#### Generators

##### Q: What is a generator? When would you use yield instead of return?
A: A generator produces values lazily — one at a time, on demand — without storing the entire sequence in memory. Critical for large files or infinite sequences. yield pauses the function and resumes from that point on the next call.

In [21]:
import glob
import pandas as pd

def load_sessions(folder):
    for path in glob.glob(f'{folder}/*.csv'):
        df = pd.read_csv(path)
        yield df

# each file loaded only when needed
for session_df in load_sessions('./'):
    print(session_df.shape)

# generator expression
magnitude = (x**2 + y**2 for x, y in zip([1,2,3], [4,5,6]))
# nothing computed yet
print(next(magnitude))
print(list(magnitude))

(44047, 11)
(9, 7)
(3000, 10)
(2000, 9)
(3001, 10)
17
[29, 45]
